In [20]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

from google_play_scraper import app, reviews, Sort 

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [21]:
CBE_APP_ID = "com.combanketh.mobilebanking"
app_info = app(CBE_APP_ID, 
               lang='en', 
               country='et')

print("=" * 50)
print("CBE Bank App Info")
print("=" * 50)
print(f"App Title: {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']}")
print(f"Total Reviews: {app_info['reviews']}")
print(f"Installs: {app_info['installs']}")


CBE Bank App Info
App Title: Commercial Bank of Ethiopia
Current Score: 4.284786
Total Ratings: 48452
Total Reviews: 9318
Installs: 5,000,000+


# Part 2 Scrapping Reviews

In [22]:
print(f"Scraping reviews for CBE Bank App...")

result, continuation_token = reviews(
    CBE_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST, #Most recent reviews first
    count=500, #Number of reviews to fetch
    filter_score_with=None #Fetch all reviews regardless of rating
)

print(f"Collected {len(result)} raw reviews.")

Scraping reviews for CBE Bank App...
Collected 500 raw reviews.


In [23]:
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"{key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
reviewId: c1e25b5d-7e79-4b60-8aa5-bed69a904f62
userName: Meti Firdu
userImage: https://play-lh.googleusercontent.com/a/ACg8ocIPZWbjpnQoosIM1RMKkt6ZLZeiaRLVE3H2gFY5jQdAgqMu_A=mo
content: worst
score: 1
thumbsUpCount: 0
reviewCreatedVersion: 5.3.0
at: 2026-05-16 12:15:55
replyContent: None
repliedAt: None
appVersion: 5.3.0


In [24]:
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review': r.get('content', ''),
        'rating': r.get('score', 0),
        'date': r.get('at', ''),
        'bank' : 'Commercial Bank of Ethiopia',
        'source' : 'Google Play Store'
    })

df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,c1e25b5d-7e79-4b60-8aa5-bed69a904f62,worst,1,2026-05-16 12:15:55,Commercial Bank of Ethiopia,Google Play Store
1,eb3cc438-1c10-4e72-8851-3efff6a04135,this app very full,5,2026-05-16 09:17:00,Commercial Bank of Ethiopia,Google Play Store
2,f8209985-ea16-4f28-bb48-d6a7276f0f08,good apps,4,2026-05-16 07:18:33,Commercial Bank of Ethiopia,Google Play Store
3,a983bc98-7ea7-4ac6-8fed-039ed2a7de0a,ok,5,2026-05-16 03:43:47,Commercial Bank of Ethiopia,Google Play Store
4,f0f249ac-ba95-4ad8-ad1d-c435693b7bf9,this update got crazy i don't know what's goin...,1,2026-05-15 23:20:32,Commercial Bank of Ethiopia,Google Play Store


## Exploring the raw data

In [25]:
#Basic Shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id            object
review               object
rating                int64
date         datetime64[ns]
bank                 object
source               object
dtype: object


In [26]:
#Rating distribution - what do users think?
print("Rating Distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"{int(rating)} stars: {count:>4} {bar}")  # Scale the bar length

Rating Distribution:
5 stars:  337 ███████████████████████████████████████████████████████████████████
4 stars:   45 █████████
3 stars:   33 ██████
2 stars:   12 ██
1 stars:   73 ██████████████


In [27]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-16 12:15:55
1   2026-05-16 09:17:00
2   2026-05-16 07:18:33
3   2026-05-16 03:43:47
4   2026-05-15 23:20:32
5   2026-05-15 20:11:22
6   2026-05-15 19:53:26
7   2026-05-15 12:22:49
8   2026-05-15 12:07:21
9   2026-05-14 18:52:51

Date dtype: datetime64[ns]


## Data Quality Audit

In [28]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("="*50)

#___ Problem 1: Missing Values ___
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"{col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
review_id      : OK
review         : OK
rating         : OK
date           : OK
bank           : OK
source         : OK


In [29]:
# Duplicate reviews
print("\nProblem 2: Duplicate")
print("-" * 30)

#Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review'], keep=False)
print(f"Exact duplicate reviews: {exact_dupes.sum()}")

#Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id'], keep=False)
print(f"Duplicate review IDs: {id_dupes.sum()}")

#Empty reviews
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"Empty reviews: {empty_reviews}")


Problem 2: Duplicate
------------------------------
Exact duplicate reviews: 146
Duplicate review IDs: 0
Empty reviews: 0


No Duplicate Id so we will not remove the duplicate reviews. 

In [30]:
# Date format issues
print("\nProblem 3: Date Format")
print("-" * 30)
print(f"Current dtype: {df_raw['date'].dtype}")
print(f" Sample values: {df_raw['date'].iloc[0]}")
print(f" Target format: YYYY-MM-DD (string or date object)")


Problem 3: Date Format
------------------------------
Current dtype: datetime64[ns]
 Sample values: 2026-05-16 12:15:55
 Target format: YYYY-MM-DD (string or date object)


# Date Cleaning


In [31]:
df = df_raw.copy()

print(f"starting with: {len(df)} reviews")

starting with: 500 reviews


In [32]:
before = len(df)

critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} reviews with missing critical data.")
print(f"Remaining: {len(df)}")

Removed 0 reviews with missing critical data.
Remaining: 500


In [33]:
before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews based on review_id.")
print(f"Remaining: {len(df)}")

Removed 0 duplicate reviews based on review_id.
Remaining: 500


In [34]:
print("Before Normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

#Convert to pandas datetime then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter Normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before Normalization:
0   2026-05-16 12:15:55
1   2026-05-16 09:17:00
2   2026-05-16 07:18:33
dtype: datetime64[ns]

After Normalization:
0    2026-05-16
1    2026-05-16
2    2026-05-16
dtype: object

Date range: 2026-03-03 to 2026-05-16


In [35]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges"""
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple spaces/newlines
    text = text.strip()
    return text

sample_raw = "   Great    app!\n\nVery useful.   "
print(f"Before: {repr(sample_raw)}")
print(f"After: {repr(clean_text(sample_raw))}")

df['review'] = df['review'].apply(clean_text)

before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"Removed {removed} reviews that were empty after cleaning.")

Before: '   Great    app!\n\nVery useful.   '
After: 'Great app! Very useful.'
Removed 0 reviews that were empty after cleaning.


In [36]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1-5): {len(invalid_ratings)}")

#Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

#Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1-5): 0
Remaining: 500 reviews
Rating dtype: int64


In [37]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

#Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,worst,1,2026-05-16,Commercial Bank of Ethiopia,Google Play Store
1,ok,5,2026-05-16,Commercial Bank of Ethiopia,Google Play Store
2,this app very full,5,2026-05-16,Commercial Bank of Ethiopia,Google Play Store
3,good apps,4,2026-05-16,Commercial Bank of Ethiopia,Google Play Store
4,thanks for you 😘,5,2026-05-15,Commercial Bank of Ethiopia,Google Play Store
5,it's okay,4,2026-05-15,Commercial Bank of Ethiopia,Google Play Store
6,It's not allowing me to transfer money.,2,2026-05-15,Commercial Bank of Ethiopia,Google Play Store
7,IT'S NOT WORK ON HUAWEI DEVICES,4,2026-05-15,Commercial Bank of Ethiopia,Google Play Store
8,this update got crazy i don't know what's goin...,1,2026-05-15,Commercial Bank of Ethiopia,Google Play Store
9,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14,Commercial Bank of Ethiopia,Google Play Store


In [38]:
#save to CSV
import os
os.makedirs('data/processed', exist_ok=True)

output_path = '../data/processed/cbe_reviews_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/cbe_reviews_reviews_clean.csv
